# Lab 3: Document Classification (Part 2)

### Overview
This topic builds on the activities of the previous topic on sentiment analysis. You will be focussing on the movie review corpus with a view to investigating the following issues.

- Evaluation metrics for classifier performance
- What is the impact of varying training data size? To what extent does increasing the quantity of training data improve classifier performance?

By this stage, you should be very comfortable with Python's [list comprehensions](http://docs.python.org/tutorial/datastructures.html#list-comprehensions) and [slice](http://bergbom.blogspot.co.uk/2011/04/python-slice-notation.html) notation.



>To access functionality defined in previous notebooks, copy the functions defined in Week3Labs into a `utils.py` file and then import it into the notebook.  There is a `utils.py` file included with these resources which you can update.

### Creating `utils.py` in Colab

Run the cell below to write the utility helper functions into `utils.py`. You can edit this cell to add, update, or paste your functions from previous weeks, then re-run it to save the changes.

In [14]:
%%writefile utils.py
# Paste your helper functions from previous labs here

class SimpleClassifier:
    def __init__(self, pos_words, neg_words):
        self.pos_words = set(pos_words)
        self.neg_words = set(neg_words)

    def classify(self, doc):
        # doc is a FreqDist of words. We count occurrences of positive and negative words.
        pos_score = sum(doc[w] for w in self.pos_words if w in doc)
        neg_score = sum(doc[w] for w in self.neg_words if w in doc)
        if pos_score >= neg_score:
            return "pos"
        else:
            return "neg"

    def classify_many(self, docs):
        return [self.classify(doc) for doc in docs]

def normalise(wordlist):
    # Add your normalization code here (e.g., lowercasing, removing stopwords/punctuation)
    return [word.lower() for word in wordlist if word.isalpha()]

def split_data(data, ratio=0.7):
    # Add your data splitting code here
    split_point = int(len(data) * ratio)
    return data[:split_point], data[split_point:]

Overwriting utils.py


In [2]:
##uncomment the following lines on colab

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#other needed imports
from nltk.corpus import movie_reviews
import random
import pandas as pd
#this next line will ensure pandas graphs are plotted in the notebook
%matplotlib inline

##download stopwords and movie_reviews - need to do this every session on colab
import nltk
nltk.download('stopwords')
nltk.download('movie_reviews')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.


True

In [4]:
#set your system path to the current working directory in Colab
import sys
sys.path.append('/content')

#import code to setup training and testing data, wordlist classifiers and NB classifiers
from utils import *


## Evaluation Metrics for Classifier Performance

### Accuracy
Here is code for an evaluation function <code>evaluate_wordlist_classifier</code> which can be used to determine how well a word_list classifier performs. This function returns the <b>accuracy</b> of a classifier. The accuracy metric is defined as the proportion of documents that were correctly classified.  Look at the code and make sure you understand what it is doing

In [5]:
def classifier_evaluate(cls, test_data):
    '''
    cls: an instance of a classifier object which has a classify method which returns "pos" or "neg"
    test_data: a list of pairs where each pair is a FreqDist rep of a doc and its label

    returns: float point number which is the accuracy of the classifier on the test data provided
    '''
    acc = 0
    docs,goldstandard=zip(*test_data) #note this neat pythonic way of turning a list of pairs into a pair of lists
    #pass all of the docs to the classifier and get back a list of predictions
    predictions=cls.classify_many(docs)
    #zip the predictions with the goldstandard labels and compare
    for prediction,goldlabel in zip(predictions,goldstandard):
        if prediction==goldlabel:
            acc+=1

    return acc / (len(test_data))

Now we need some data to train and test our classifier on.  We are going to make use of the code from the lab_3_1.  

In [13]:
def get_train_test_data():
    # STEP 1: Retrieve the lists of file names (IDs) for positive and negative reviews from NLTK
    pos_review_ids = movie_reviews.fileids('pos')
    neg_review_ids = movie_reviews.fileids('neg')

    # STEP 2: Split both positive and negative review files into training (70%) and testing (30%) sets
    # (Using the split_data helper function we imported from utils.py)
    pos_train_ids, pos_test_ids = split_data(pos_review_ids)
    neg_train_ids, neg_test_ids = split_data(neg_review_ids)

    # STEP 3: Load the actual words for each review file and label them ('pos' or 'neg')
    # This creates lists of pairs like: ([list, of, words], 'pos')
    training = [(movie_reviews.words(f), 'pos') for f in pos_train_ids] + [(movie_reviews.words(f), 'neg') for f in neg_train_ids]
    testing = [(movie_reviews.words(f), 'pos') for f in pos_test_ids] + [(movie_reviews.words(f), 'neg') for f in neg_test_ids]

    # STEP 4: Clean the text (normalization) and convert them to word frequency counts (Bag-of-Words)
    # For each review, we lowercase/remove punctuation, and then build a Frequency Distribution (FreqDist)
    training_norm = [(FreqDist(normalise(wordlist)), label) for (wordlist, label) in training]
    testing_norm = [(FreqDist(normalise(wordlist)), label) for (wordlist, label) in testing]

    # STEP 5: Return the fully prepared training and testing sets
    return training_norm, testing_norm

In [15]:
from nltk import FreqDist

random.seed(41)  #set the random seeds so these random splits are always the same
training,testing=get_train_test_data()

training[0]

(FreqDist({'the': 46, 'and': 21, 'in': 18, 'a': 15, 'to': 15, 'of': 14, 's': 11, 'it': 11, 't': 10, 'from': 8, ...}),
 'pos')

We are now going to use this function to start evaluating our classifiers from last week.  Lets first try the SimpleClassifier

In [17]:
# The original SimpleClassifier assigns everything to the positive class by default.
# If we instantiate it with empty lists, both pos_score and neg_score will be 0.
# Since pos_score >= neg_score (0 >= 0) is True, it will classify everything as "pos".

baseline_classifier = SimpleClassifier(pos_words=[], neg_words=[])

# Evaluate the classifier on our testing data
baseline_accuracy = classifier_evaluate(baseline_classifier, testing)
print(f"Baseline Classifier Accuracy: {baseline_accuracy}")

Baseline Classifier Accuracy: 0.5


If you have run the cell above without updating the SimpleClassifier code you should see that the accuracy is 0.5 i.e., 50%. The original SimpleClassifier just assigns everything to the positive class.  Since it is a binary classification decision and the classes are balanced, it will get 50% of the decisions correct (those that are positive) and 50% of the decisions incorrect (those that are actually negative).  This is the **baseline** result for this kind of classification task.  We obviously want to build classifiers that do better than this.

### Exercise 1

Now you try one of the classifiers that you wrote that selects positive and negative words from the training data.  Hopefully, this will perform better than the baseline of 50%.

In [18]:
# Force reload the utils module to import the newly written SimpleClassifier
import importlib
import utils
importlib.reload(utils)
from utils import SimpleClassifier

#here I am going to create an instance of a very simple classifier
my_positive_word_list = ["good","great","lovely", "excellent","amazing","wonderful"] # extend this one or put your own list here
my_negative_word_list = ["bad", "terrible", "awful", "dreadful","shit","crap"] # extend this one or put your own list here
movie_classifier1 = SimpleClassifier(my_positive_word_list,my_negative_word_list)

#Evaluate classifier
#The function requires two arguments:
# 1. Word list based classifer
# 2. A list (or generator) of labelled test items
score = classifier_evaluate(movie_classifier1, testing)
print(score)

0.625


## Precision, Recall and F1 score etc

When classes are unbalanced, evaluating classifiers in terms of accuracy can be misleading.  For example, if 10% of documents are relevant and 90% of documents are irrelevant, then a classifier which labels all documents as irrelevant will obtain an accuracy of 90%.  This sounds good but is actually useless. More useful metrics for evaluation of performance are precision, recall and F1 score.  These metrics allow us to distinguish the different types of errors our classifiers make.

For each class, $c$, we need to keep a record of
* True Positives: $TP=|\{i|\mbox{prediction}(i)=\mbox{label}(i)=c\}|$
* False Negatives: $FN=|\{i|\mbox{prediction}(i)\neq \mbox{label}(i)=c\}|$
* False Positives: $FP=|\{i|\mbox{label}(i) \neq \mbox{prediction}(i)=c\}|$
* True Negatives: $TN=|\{i|\mbox{prediction}(i)=\mbox{label}(i)\neq c\}|$

Note the symmetry in the binary classification task (the TN for one class are the TP for the other class and so on).  Therefore, in binary classification, we just record these values and compute the following evaluation metrics for a single class (e.g. "Relevant" or "Positive")

* Precision:
\begin{eqnarray*}
P=\frac{TP}{TP+FP}
\end{eqnarray*}
* Recall:
\begin{eqnarray*}
R=\frac{TP}{TP+FN}
\end{eqnarray*}
* F1-score:
\begin{eqnarray*}
F1 = \frac{2\times P\times R}{P+R}
\end{eqnarray*}


 ### Exercise 2.1

 The code below defines a ConfusionMatrix class for the binary classification task.  Currently, it will compute the number of TPs, FPs, FNs and TNs.  Test it out with predictions and test data for our sentiment analysis task.

In [19]:
class ConfusionMatrix:
    def __init__(self,predictions,goldstandard,classes=("pos","neg")):

        (self.c1,self.c2)=classes
        self.TP=0
        self.FP=0
        self.FN=0
        self.TN=0
        for p,g in zip(predictions,goldstandard):
            if g==self.c1:
                if p==self.c1:
                    self.TP+=1
                else:
                    self.FN+=1

            elif p==self.c1:
                self.FP+=1
            else:
                self.TN+=1


def precision(self):
    # "Of everything I predicted as pos, how many really were pos?"
    # TP + FP = every item the model labelled pos (right or wrong)
    if (self.TP + self.FP) == 0:
        # The model never predicted pos, so there is nothing to divide by.
        # Return 0.0 rather than crash with a ZeroDivisionError.
        return 0.0
    return self.TP / (self.TP + self.FP)

def recall(self):
    # "Of everything that really was pos, how many did I find?"
    # TP + FN = every item that is pos in the gold standard
    # (FN = real positives the model missed)
    if (self.TP + self.FN) == 0:
        # There are no real positives in the data, so avoid dividing by zero.
        return 0.0
    return self.TP / (self.TP + self.FN)

def f1(self):
    # One score that balances precision and recall.
    p = self.precision()
    r = self.recall()
    if (p + r) == 0:
        # Both are 0 (no true positives at all), so avoid dividing by zero.
        return 0.0
    # Harmonic mean: it is dragged down by whichever of p or r is lower,
    # so you only score well if BOTH are good.
    return (2 * p * r) / (p + r)

### Exercise 2.2
* Add functionality to the ConfusionMatrix class code to compute precision, recall and F1 score
* Use your code to evaluate the performance of the different classifiers you have constructed.
* Interpret your results

## Investigating the impact of the quantity of training data
We will begin by exploring the impact on classification accuracy of using different quantities of training data.




Run the code in the cell below several times.  Each time it should generate a new sample of review data, train the classifiers and evaluate them.

In [ ]:

training,testing=get_train_test_data()

word_list_size = 100
threshold=100
classifiers={"Word List MF":SimpleClassifier_mf(word_list_size),
             "Word List Thresh":SimpleClassifier_ot(threshold)}

results=[]
for name,classifier in classifiers.items():
    classifier.train(training)
    accuracy=classifier_evaluate(classifier,testing)
    print("The accuracy of {} classifier is {}".format(name,accuracy))
    results.append((name,accuracy))

df = pd.DataFrame(results)
display(df)
ax = df.plot.bar(title="Experimental Results",legend=False,x=0)
ax.set_ylabel("Classifier Accuracy")
ax.set_xlabel("Classifier")
ax.set_ylim(0,1.0)

As you can see, the classifiers have different accuracies on different runs.

### Exercise 3.1
Copy the cell above and move the copy to be positioned below this cell. Then adapt the code so that the accuracy reported for each classifier is the average across multiple runs.

### Exercise 3.2
Adapt the code so that it calculates average precision, recall and F1-score rather than average accuracy.

### Exercise 3.3
Investigate the impact of training data size on the performance of both word list classifiers, using a range of subsets of movie reviews.

Hints and tips.
- You can copy the code cell that you created for the last exercise, and place the copy below this cell. Then adapt the code to determine accuracy, precision, recall and F1-score for each classifier on each subset.

- Use the `sample` function from the random module.
- Remember, the full data set has 1000 positive and 1000 negative reviews.
- You should continue to use 30% of the data for testing, so this means that we have up to 700 positive and 700 negative reviews to sample from.
- Make sure that you are selecting samples that have an equal number of positive and negative reviews.
- Consider (at least) the following sample sizes: 2, 10, 50, 100, 200, 400, 600 and 700.
- Note that the sample size is not the total number of reviews, but the number of positive reviews (which is also equal to the number of negative reviews).


**Interpret your results**

My answer:

